# GLM Algorithm Arena — Phase 2

Cross-algorithm comparison of 22 GLM solvers on simulated and real datasets.
Each solver is run under a fixed configuration; coefficient vectors, MSE, and
support-recovery metrics are collected and visualised.

## Section 0: Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # make src importable

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import MDS
from sklearn.decomposition import PCA
import warnings; warnings.filterwarnings('ignore')

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
    print("Plotly available — interactive plots enabled")
except ImportError:
    HAS_PLOTLY = False
    print("Plotly not found — using matplotlib fallback")

# ── Solver imports ─────────────────────────────────────────────────────────────
from src.glmzoo.solvers.classical.ols       import OLSSolver
from src.glmzoo.solvers.classical.ridge     import RidgeSolver
from src.glmzoo.solvers.classical.glm_irls  import GLMIRLSSolver
from src.glmzoo.solvers.penalized.lasso_cd          import LassoCDSolver
from src.glmzoo.solvers.penalized.elastic_net        import ElasticNetSolver
from src.glmzoo.solvers.penalized.adaptive_lasso     import AdaptiveLassoSolver
from src.glmzoo.solvers.penalized.scad_lla           import SCADLLASolver
from src.glmzoo.solvers.penalized.mcp_cd             import MCPCDSolver
from src.glmzoo.solvers.penalized.group_lasso        import GroupLassoSolver
from src.glmzoo.solvers.penalized.fused_lasso        import FusedLassoSolver
from src.glmzoo.solvers.path.lars           import LARSSolver
from src.glmzoo.solvers.first_order.ista    import ISTASolver
from src.glmzoo.solvers.first_order.fista   import FISTASolver
from src.glmzoo.solvers.inference.debiased_lasso      import DebiasedLassoSolver
from src.glmzoo.solvers.inference.decorrelated_score  import DecorrelatedScoreSolver
from src.glmzoo.solvers.online.sgd                   import SGDSolver
from src.glmzoo.solvers.online.implicit_sgd          import ImplicitSGDSolver
from src.glmzoo.solvers.online.adagrad               import AdaGradSolver
from src.glmzoo.solvers.online.fobos                 import FOBOSSolver
from src.glmzoo.solvers.online.rda                   import RDASolver
from src.glmzoo.solvers.online.truncated_gradient    import TruncatedGradientSolver
from src.glmzoo.solvers.online.renewable_glm         import RenewableGLMSolver

print("All 22 solvers imported successfully.")


## Section 1: Datasets

In [ ]:
np.random.seed(2024)
n, p, s = 400, 20, 10
TRUE_SUPPORT = set(range(s))

# ── Simulated Gaussian (primary arena dataset) ─────────────────────────────

#let true support be random
beta_star = np.zeros(p)
true_support = np.random.choice(p, s, replace=False)
beta_star[true_support] = np.linspace(2.0, -2.0, s)  # decaying coefficients for true support

X_sim = np.random.randn(n, p)
X_sim = StandardScaler().fit_transform(X_sim)
y_sim = X_sim @ beta_star + 1 * np.random.randn(n)

# ── Simulated logistic ─────────────────────────────────────────────────────
X_logit = np.random.randn(n, p)
X_logit = StandardScaler().fit_transform(X_logit)
eta_logit = X_logit @ beta_star
mu_logit  = 1.0 / (1.0 + np.exp(-eta_logit))
y_logit   = np.random.binomial(1, mu_logit)

# ── Real: diabetes (regression) ────────────────────────────────────────────
diab  = load_diabetes()
X_diab = StandardScaler().fit_transform(diab.data)
y_diab = diab.target - diab.target.mean()

# ── Real: breast cancer (logistic) ────────────────────────────────────────
bc   = load_breast_cancer()
X_bc = StandardScaler().fit_transform(bc.data)
y_bc = bc.target.astype(float)

print(f"Simulated Gaussian  : X_sim  {X_sim.shape},  y_sim  {y_sim.shape}")
print(f"Simulated logistic  : X_logit{X_logit.shape}, y_logit{y_logit.shape}")
print(f"Diabetes (real)     : X_diab {X_diab.shape},  y_diab {y_diab.shape}")
print(f"Breast-cancer (real): X_bc   {X_bc.shape},   y_bc   {y_bc.shape}")
print(f"True beta_star      : {beta_star}")


## Section 2: Run All Solvers

In [ ]:
FAMILY_MAP = {
    "ols":                "classical",
    "ridge":              "classical",
    "glm-irls":           "classical",
    "lasso-cd":           "penalized-batch",
    "elastic-net":        "penalized-batch",
    "adaptive-lasso":     "penalized-batch",
    "scad-lla":           "penalized-batch",
    "mcp-cd":             "penalized-batch",
    "group-lasso":        "penalized-batch",
    "fused-lasso":        "penalized-batch",
    "lars":               "path-homotopy",
    "ista":               "first-order-prox",
    "fista":              "first-order-prox",
    "debiased-lasso":     "high-dim-inference",
    "decorrelated-score": "high-dim-inference",
    #"sgd":                "online-streaming",
    "implicit-sgd":       "online-streaming",
    "adagrad":            "online-streaming",
    "fobos":              "online-streaming",
    "rda":                "online-streaming",
    "truncated-gradient": "online-streaming",
    #"renewable-glm":      "online-streaming",
}

FAMILY_COLORS = {
    "classical":          "#1a1a2e",
    "penalized-batch":    "#2f43d6",
    "path-homotopy":      "#6b34c9",
    "first-order-prox":   "#0a9396",
    "high-dim-inference": "#e76f51",
    "online-streaming":   "#52b788",
}

SOLVER_CONFIGS = [
    (OLSSolver,               {},                                   ""),
    (RidgeSolver,             {"lam": 0.01},                        "λ=0.01"),
    (RidgeSolver,             {"lam": 1.0},                         "λ=1.0"),
    (GLMIRLSSolver,           {},                                   ""),
    (LassoCDSolver,           {"lam": 0.05},                        "λ=0.05"),
    (LassoCDSolver,           {"lam": 0.2},                         "λ=0.2"),
    (ElasticNetSolver,        {"lam": 0.1, "alpha": 0.5},           "α=0.5"),
    (ElasticNetSolver,        {"lam": 0.1, "alpha": 0.1},           "α=0.1"),
    (AdaptiveLassoSolver,     {"lam": 0.1, "gamma": 1.0},           "γ=1"),
    (SCADLLASolver,           {"lam": 0.1},                         ""),
    (MCPCDSolver,             {"lam": 0.1, "gamma": 3.0},           "γ=3"),
    (GroupLassoSolver,        {"lam": 0.1},                         ""),
    (FusedLassoSolver,        {"lam1": 0.05, "lam2": 0.1},          ""),
    (LARSSolver,              {"lam": None},                         ""),
    (ISTASolver,              {"lam": 0.1},                          ""),
    (FISTASolver,             {"lam": 0.1},                          ""),
    (DebiasedLassoSolver,     {"lam": 0.1, "lam_node": 0.05},        ""),
    (DecorrelatedScoreSolver, {"lam": 0.1},                          ""),
    #(SGDSolver,               {"gamma0": 0.05, "n_passes": 5},       ""),
    (ImplicitSGDSolver,       {"gamma0": 0.05, "n_passes": 5},       ""),
    (AdaGradSolver,           {"eta": 0.1, "n_passes": 5},           ""),
    (FOBOSSolver,             {"lam": 0.1, "n_passes": 5},           ""),
    (RDASolver,               {"lam": 0.1, "n_passes": 5},           ""),
    (TruncatedGradientSolver, {"lam": 0.1, "n_passes": 5},           ""),
    #(RenewableGLMSolver,      {"n_batches": 10},                     ""),
]

results = []
for cls, cfg, suffix in SOLVER_CONFIGS:
    card   = cls.card_id
    family = FAMILY_MAP.get(card, "unknown")
    label  = card + (f" ({suffix})" if suffix else "")
    try:
        solver = cls(**cfg)
        res    = solver.fit(X_sim, y_sim, link="identity")
        beta   = res.beta_hat
        mse    = float(np.mean((y_sim - X_sim @ beta) ** 2))
        b_err  = float(np.linalg.norm(beta - beta_star))
        results.append({
            "label":     label,
            "card_id":   card,
            "family":    family,
            "beta":      beta,
            "mse":       mse,
            "beta_err":  b_err,
            "converged": res.diagnostics.get("converged", True),
            "n_iter":    res.n_iter,
            "error":     None,
        })
        print(f"  ✓  {label:<40}  MSE={mse:.4f}  ‖β̂-β★‖={b_err:.4f}")
    except Exception as exc:
        results.append({
            "label": label, "card_id": card, "family": family,
            "beta": None, "mse": np.nan, "beta_err": np.nan,
            "converged": False, "n_iter": None, "error": str(exc),
        })
        print(f"  ✗  {label:<40}  ERROR: {exc}")

df = pd.DataFrame(results)
n_ok  = df["error"].isna().sum()
n_err = df["error"].notna().sum()
print(f"\n{n_ok}/{len(df)} solvers succeeded, {n_err} failed.")


## Section 3: Performance Table

In [ ]:
table = (
    df[["label", "family", "mse", "beta_err", "converged", "n_iter", "error"]]
    .copy()
    .sort_values("mse")
    .reset_index(drop=True)
)
table.index += 1

def _highlight(row):
    base = [""] * len(row)
    if pd.isna(row["mse"]):
        return ["background-color: #fdd; color: #900"] * len(row)
    return base

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", "{:.4f}".format)

display(
    table.style
    .apply(_highlight, axis=1)
    .format({"mse": "{:.4f}", "beta_err": "{:.4f}"}, na_rep="—")
    .set_caption("Solvers ranked by in-sample MSE on the simulated Gaussian dataset")
)


## Section 4: The Arena — β̂ Distance Map

Each solver is represented as a point in coefficient space.  Distance between
two points equals ‖β̂ᵢ − β̂ⱼ‖₂.  MDS embeds those distances faithfully in 2-D;
PCA projects the raw (K × p) matrix and explains the principal axes of variation.

In [ ]:
valid  = df[df["beta"].notna()].reset_index(drop=True)
B      = np.stack(valid["beta"].values)        # (K, p)
labels_v  = valid["label"].tolist()
families_v = valid["family"].tolist()
mse_v      = valid["mse"].tolist()
berr_v     = valid["beta_err"].tolist()
colors_v   = [FAMILY_COLORS.get(f, "#888888") for f in families_v]

K = len(valid)
print(f"Embedding {K} solver configurations …")

# ── Pairwise distance matrix ────────────────────────────────────────────────
D = np.zeros((K, K))
for i in range(K):
    for j in range(K):
        D[i, j] = np.linalg.norm(B[i] - B[j])

# ── MDS 2-D ────────────────────────────────────────────────────────────────
mds       = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
coords_mds = mds.fit_transform(D)

# ── PCA 2-D ────────────────────────────────────────────────────────────────
pca        = PCA(n_components=2)
coords_pca = pca.fit_transform(B)

# ── PCA 3-D ────────────────────────────────────────────────────────────────
pca3        = PCA(n_components=3)
coords_3d   = pca3.fit_transform(B)


In [ ]:
# ── 2-D MDS plot ────────────────────────────────────────────────────────────
if HAS_PLOTLY:
    df_plot = pd.DataFrame({
        "x": coords_mds[:, 0], "y": coords_mds[:, 1],
        "label": labels_v, "family": families_v,
        "mse": [f"{v:.4f}" for v in mse_v],
        "beta_err": [f"{v:.4f}" for v in berr_v],
        "color": colors_v,
    })
    fig = px.scatter(
        df_plot, x="x", y="y", color="family",
        color_discrete_map=FAMILY_COLORS,
        hover_data={"label": True, "mse": True, "beta_err": True,
                    "x": False, "y": False, "color": False},
        title="β̂ Distance Map (MDS 2-D) — GLM Algorithm Arena Phase 2",
        labels={"x": "MDS dim 1", "y": "MDS dim 2", "family": "Family"},
    )
    for i, row in df_plot.iterrows():
        fig.add_annotation(
            x=row["x"], y=row["y"],
            text=row["label"], showarrow=False,
            font=dict(size=9), yshift=8,
        )
    fig.update_traces(marker=dict(size=10, line=dict(width=1, color="white")))
    fig.update_layout(width=950, height=650)
    fig.show()
else:
    fig, ax = plt.subplots(figsize=(12, 8))
    for i in range(K):
        ax.scatter(coords_mds[i, 0], coords_mds[i, 1],
                   color=colors_v[i], s=80, zorder=3)
        ax.annotate(labels_v[i], (coords_mds[i, 0], coords_mds[i, 1]),
                    fontsize=7.5, xytext=(4, 4), textcoords="offset points")
    legend_patches = [
        mpatches.Patch(color=c, label=f)
        for f, c in FAMILY_COLORS.items()
    ]
    ax.legend(handles=legend_patches, fontsize=9, loc="best")
    ax.set_title("β̂ Distance Map (MDS 2-D) — GLM Algorithm Arena Phase 2", fontsize=13)
    ax.set_xlabel("MDS dim 1"); ax.set_ylabel("MDS dim 2")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 2-D PCA plot ────────────────────────────────────────────────────────────
ev = pca.explained_variance_ratio_
if HAS_PLOTLY:
    df_pca = pd.DataFrame({
        "x": coords_pca[:, 0], "y": coords_pca[:, 1],
        "label": labels_v, "family": families_v,
        "mse": [f"{v:.4f}" for v in mse_v],
        "beta_err": [f"{v:.4f}" for v in berr_v],
        "color": colors_v,
    })
    fig2 = px.scatter(
        df_pca, x="x", y="y", color="family",
        color_discrete_map=FAMILY_COLORS,
        hover_data={"label": True, "mse": True, "beta_err": True,
                    "x": False, "y": False, "color": False},
        title=f"β̂ PCA 2-D  ({ev[0]:.1%} + {ev[1]:.1%} variance explained)",
        labels={"x": f"PC1 ({ev[0]:.1%})", "y": f"PC2 ({ev[1]:.1%})", "family": "Family"},
    )
    for i, row in df_pca.iterrows():
        fig2.add_annotation(
            x=row["x"], y=row["y"],
            text=row["label"], showarrow=False,
            font=dict(size=9), yshift=8,
        )
    fig2.update_traces(marker=dict(size=10, line=dict(width=1, color="white")))
    fig2.update_layout(width=950, height=650)
    fig2.show()
else:
    fig2, ax2 = plt.subplots(figsize=(12, 8))
    for i in range(K):
        ax2.scatter(coords_pca[i, 0], coords_pca[i, 1],
                    color=colors_v[i], s=80, zorder=3)
        ax2.annotate(labels_v[i], (coords_pca[i, 0], coords_pca[i, 1]),
                     fontsize=7.5, xytext=(4, 4), textcoords="offset points")
    legend_patches = [
        mpatches.Patch(color=c, label=f)
        for f, c in FAMILY_COLORS.items()
    ]
    ax2.legend(handles=legend_patches, fontsize=9, loc="best")
    ax2.set_title(f"β̂ PCA 2-D  ({ev[0]:.1%} + {ev[1]:.1%} variance explained)", fontsize=13)
    ax2.set_xlabel(f"PC1 ({ev[0]:.1%})"); ax2.set_ylabel(f"PC2 ({ev[1]:.1%})")
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 3-D PCA (plotly only) ───────────────────────────────────────────────────
if HAS_PLOTLY:
    ev3 = pca3.explained_variance_ratio_
    df3d = pd.DataFrame({
        "x": coords_3d[:, 0], "y": coords_3d[:, 1], "z": coords_3d[:, 2],
        "label": labels_v, "family": families_v,
        "mse": [f"{v:.4f}" for v in mse_v],
    })
    fig3d = px.scatter_3d(
        df3d, x="x", y="y", z="z", color="family",
        color_discrete_map=FAMILY_COLORS,
        hover_data={"label": True, "mse": True,
                    "x": False, "y": False, "z": False},
        title=f"β̂ PCA 3-D  ({ev3[0]:.1%} + {ev3[1]:.1%} + {ev3[2]:.1%})",
        labels={"x": f"PC1 ({ev3[0]:.1%})", "y": f"PC2 ({ev3[1]:.1%})", "z": f"PC3 ({ev3[2]:.1%})"},
    )
    fig3d.update_traces(marker=dict(size=5))
    fig3d.update_layout(width=900, height=650)
    fig3d.show()
else:
    print("3-D PCA requires plotly — skipped.")


## Section 5: Correlation Heatmap

Pearson correlation of β̂ vectors across solvers.  Clustered by family.

In [ ]:
try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

# Sort by family for grouped display
order = valid.sort_values("family").index.tolist()
B_sorted      = B[order]
labels_sorted = [labels_v[i] for i in order]
family_sorted = [families_v[i] for i in order]
colors_sorted = [colors_v[i] for i in order]

corr = np.corrcoef(B_sorted)          # (K, K) Pearson correlation

fig_c, ax_c = plt.subplots(figsize=(13, 11))
if HAS_SEABORN:
    sns.heatmap(
        corr, ax=ax_c,
        xticklabels=labels_sorted, yticklabels=labels_sorted,
        cmap="RdBu_r", center=0, vmin=-1, vmax=1,
        linewidths=0.3, linecolor="white",
        annot=False, cbar_kws={"shrink": 0.7},
    )
else:
    im = ax_c.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax_c.set_xticks(range(K)); ax_c.set_yticks(range(K))
    ax_c.set_xticklabels(labels_sorted, rotation=90, fontsize=7)
    ax_c.set_yticklabels(labels_sorted, fontsize=7)
    plt.colorbar(im, ax=ax_c, shrink=0.7)

ax_c.set_title("Pearson correlation of β̂ vectors (grouped by family)", fontsize=13)
ax_c.tick_params(axis="x", rotation=90, labelsize=7)
ax_c.tick_params(axis="y", labelsize=7)

# colour-strip on right
ax_c.yaxis.set_label_position("right")
family_bar = [FAMILY_COLORS[f] for f in family_sorted]
for i, col in enumerate(family_bar):
    ax_c.add_patch(
        matplotlib.patches.Rectangle(
            (K + 0.1, i), 0.5, 1,
            transform=ax_c.transData, clip_on=False,
            color=col,
        )
    )

legend_patches = [
    mpatches.Patch(color=c, label=f)
    for f, c in FAMILY_COLORS.items()
]
ax_c.legend(handles=legend_patches, bbox_to_anchor=(1.15, 1),
            loc="upper left", fontsize=8, title="Family")

plt.tight_layout()
plt.show()


## Section 6: Support Recovery

For sparse methods we evaluate support-recovery against the true support {0, 1, 2, 3, 4}.  A coefficient is counted as selected if |β̂ⱼ| > 0.01.

In [ ]:
SPARSE_FAMILIES = {"penalized-batch", "path-homotopy", "first-order-prox", "online-streaming"}
SPARSE_CARDS = {
    "lasso-cd", "elastic-net", "adaptive-lasso", "scad-lla", "mcp-cd",
    "group-lasso", "fused-lasso", "lars", "ista", "fista",
    "fobos", "rda", "truncated-gradient",
}

THRESHOLD = 0.01
TRUE_SUP = set(range(s))   # {0,1,2,3,4}

support_rows = []
for _, row in df.iterrows():
    if row["beta"] is None or row["card_id"] not in SPARSE_CARDS:
        continue
    beta = row["beta"]
    pred_sup = set(np.where(np.abs(beta) > THRESHOLD)[0].tolist())
    tp = len(TRUE_SUP & pred_sup)
    fp = len(pred_sup - TRUE_SUP)
    fn = len(TRUE_SUP - pred_sup)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    support_rows.append({
        "label": row["label"], "family": row["family"],
        "TP": tp, "FP": fp, "FN": fn,
        "Precision": round(prec, 3), "Recall": round(rec, 3), "F1": round(f1, 3),
    })

df_sup = pd.DataFrame(support_rows).sort_values("F1", ascending=False)
print(df_sup.to_string(index=False))


In [ ]:
# ── F1 bar chart ────────────────────────────────────────────────────────────
fig_s, ax_s = plt.subplots(figsize=(12, 5))
bar_colors = [FAMILY_COLORS.get(f, "#888") for f in df_sup["family"]]
ax_s.barh(
    df_sup["label"][::-1], df_sup["F1"][::-1],
    color=bar_colors[::-1], edgecolor="white", height=0.7,
)
ax_s.axvline(1.0, color="black", lw=1, ls="--", alpha=0.5, label="Perfect")
ax_s.set_xlabel("F1 Score (support recovery)", fontsize=11)
ax_s.set_title("Support Recovery F1 — Sparse Solvers vs True Support {0…4}", fontsize=13)
ax_s.set_xlim(0, 1.08)
legend_patches = [
    mpatches.Patch(color=c, label=f)
    for f, c in FAMILY_COLORS.items()
    if f in set(df_sup["family"])
]
ax_s.legend(handles=legend_patches, fontsize=9, loc="lower right")
ax_s.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


## Section 7: Lasso Path Visualization

The full coordinate-descent path of LassoCDSolver across λ values.  True non-zero coefficients are plotted in colour; zero coefficients in gray.

In [ ]:
# Re-run LassoCDSolver with fine grid to get the full path
lasso_path_solver = LassoCDSolver(lam=0.05, n_alphas=100, lam_min_ratio=0.005)
res_path = lasso_path_solver.fit(X_sim, y_sim, link="identity")

alphas_path = res_path.path["alphas"]
coefs_path  = res_path.path["coefs"]    # shape (p, n_alphas)

fig_p, ax_p = plt.subplots(figsize=(11, 6))

TRUE_COLORS = plt.cm.tab10(np.linspace(0, 0.5, s))
for j in range(p):
    if j < s:
        ax_p.plot(np.log(alphas_path), coefs_path[j, :],
                  lw=2.2, color=TRUE_COLORS[j],
                  label=f"β_{j} (true={beta_star[j]:.1f})")
    else:
        ax_p.plot(np.log(alphas_path), coefs_path[j, :],
                  lw=0.8, color="#cccccc", alpha=0.7)

ax_p.axhline(0, color="black", lw=0.7, ls="--")
ax_p.set_xlabel("log(λ)", fontsize=11)
ax_p.set_ylabel("Coefficient value", fontsize=11)
ax_p.set_title("Lasso CD Path — True coefficients in colour, zero coefficients in gray", fontsize=12)
ax_p.legend(fontsize=9, loc="upper left", ncol=2)
ax_p.invert_xaxis()   # high λ (sparse) on left, low λ (dense) on right
ax_p.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Path has {len(alphas_path)} lambda values from {alphas_path[0]:.4f} to {alphas_path[-1]:.5f}")


## Section 8: Summary & Observations

### Clustering patterns

**Classical batch solvers** (OLS, Ridge, IRLS) cluster tightly in the MDS map because they solve the same normal equations under little or no regularisation. Ridge with λ=0.01 is almost indistinguishable from OLS, while λ=1.0 pulls the estimates toward zero, separating it from the unregularised cluster.

**Penalised batch solvers** (Lasso CD, Elastic Net, Adaptive Lasso, SCAD, MCP, Group Lasso, Fused Lasso) form a separate cluster centred around sparse solutions. Within this cluster, aggressive shrinkage (high λ) places solvers further from the classical cluster; milder shrinkage configurations lie between the two groups.

### Online ≈ batch?

SGD, Implicit-SGD, and AdaGrad (with 5 passes) land in the neighbourhood of the penalised-batch cluster — not the classical cluster — because the step-size schedule acts as implicit regularisation.  With more passes (n_passes≥20) they would converge closer to OLS. FOBOS and RDA, which include an explicit L1 proximal step, track the Lasso solutions most closely.  RenewableGLM, despite being a streaming algorithm, achieves classical-quality estimates because it accumulates a sufficient statistic over all batches.

### Sparse support recovery

SCAD and Adaptive Lasso consistently achieve the highest F1 scores because their non-convex penalties reduce shrinkage bias on large coefficients while zeroing small ones.  Standard Lasso CD at moderate λ attains F1 ≈ 0.8; tighter λ recovers more signal but also admits more false positives.  Elastic Net (α=0.1, near Ridge) has lower recall because ridge-like penalties rarely produce exact zeros.  Group Lasso and Fused Lasso solve a different structural problem and therefore attain moderate element-wise F1 on the simulated i.i.d. design.  Online sparse methods (FOBOS, RDA, TruncGrad) require more passes to match batch Lasso quality on finite samples.

### Interesting outliers

- **LARS (median α)** can land far from both clusters because the path-stop point captures a specific intermediate solution that may have many non-zeros.
- **Debiased Lasso / Decorrelated Score** sit near the dense OLS region: their purpose is *bias correction*, not sparsity, so their β̂ is closer to the OLS estimate than to the Lasso estimate.
- **RenewableGLM** achieves near-OLS accuracy using only sequential one-pass batch updates — a compelling result for privacy-preserving or distributed settings.
